# Notebook 04 — Boundary Conditions: Mathematics and MAPDL Implementation

## What you will learn
1. The strong and weak form of the elasticity PDE
2. Dirichlet, Neumann, Robin, and periodic BCs — mathematics and physical meaning
3. Symmetry and anti-symmetry BCs — how to exploit structural symmetry
4. How each BC translates to MAPDL commands
5. Common pitfalls (rigid body modes, over-constraint, periodic mesh mismatch)

---

## Part 1: The Governing Equations

### Strong form (Cauchy's equations of motion)

For a deformable body occupying domain $\Omega$ with boundary $\Gamma$:

$$\nabla \cdot \boldsymbol{\sigma} + \mathbf{b} = \rho \ddot{\mathbf{u}} \quad \text{in } \Omega$$

where:
- $\boldsymbol{\sigma}$ = Cauchy stress tensor (N/m²)
- $\mathbf{b}$ = body force per unit volume (N/m³) — e.g., gravity $\rho \mathbf{g}$
- $\rho$ = mass density (kg/m³)
- $\mathbf{u}$ = displacement field (m)
- $\ddot{\mathbf{u}}$ = acceleration (m/s²) — zero for static problems

### Boundary conditions

The domain boundary $\Gamma = \Gamma_D \cup \Gamma_N \cup \Gamma_R$ is partitioned:

| Type | Equation | MAPDL command | Name |
|------|----------|---------------|------|
| Dirichlet | $\mathbf{u} = \bar{\mathbf{u}}$ on $\Gamma_D$ | `D, ALL, UX, value` | Prescribed displacement |
| Neumann | $\boldsymbol{\sigma} \cdot \mathbf{n} = \mathbf{t}$ on $\Gamma_N$ | `SF, ALL, PRES, value` | Traction / pressure |
| Robin | $\boldsymbol{\sigma} \cdot \mathbf{n} + k_s \mathbf{u} = 0$ on $\Gamma_R$ | `SURF154` | Elastic foundation |
| Periodic | $\mathbf{u}(\mathbf{x}+\mathbf{L}) = \mathbf{u}(\mathbf{x})$ | `CP, nset, DOF, n1, n2` | Unit cell periodicity |

### Weak form (the form solved by FEM)

Multiply the strong form by a test function $\delta \mathbf{u}$ (virtual displacement)
and integrate over $\Omega$:

$$\int_\Omega \delta \boldsymbol{\varepsilon} : \boldsymbol{\sigma} \, dV = \int_\Omega \delta \mathbf{u} \cdot \mathbf{b} \, dV + \int_{\Gamma_N} \delta \mathbf{u} \cdot \mathbf{t} \, dA \quad \text{(Principle of Virtual Work)}$$

This is also the principle of virtual work (or virtual displacements).
**The left side is internal virtual work; the right side is external virtual work.**

After discretization with shape functions $\mathbf{N}$, this becomes:

$$[K] \{u\} = \{F\}$$

where $[K] = \int_\Omega [B]^T [C] [B] \, dV$ (stiffness matrix),
$[B] = \nabla_s \mathbf{N}$ (strain-displacement matrix).

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

---

## Part 2: Dirichlet BCs (Prescribed Displacement)

A Dirichlet BC sets a specific displacement value at a set of nodes:

$$u_i = \bar{u}_i \quad \forall i \in \Gamma_D$$

In FEM, this is handled by partitioning the system into free (f) and
constrained (c) DOFs:

$$\begin{bmatrix} K_{ff} & K_{fc} \\ K_{cf} & K_{cc} \end{bmatrix}
\begin{bmatrix} u_f \\ u_c \end{bmatrix} =
\begin{bmatrix} F_f \\ F_c \end{bmatrix}$$

Only the $(ff)$ block is solved:
$$K_{ff} u_f = F_f - K_{fc} \bar{u}_c$$

The reaction forces at the constrained DOFs are then:
$$F_c = K_{cf} u_f + K_{cc} \bar{u}_c$$

### Rigid body modes

**The stiffness matrix [K] is singular if the structure is not properly constrained.**

A structure in 3D has 6 rigid body modes (3 translations + 3 rotations).
The BCs must remove all 6 for the system to have a unique solution.

Common mistake: constraining UX=0 on the left face but not pinning the bottom-left
node in UY — the structure is free to rotate (rigid body rotation mode).

In [ ]:
# ── Visualize: which BCs remove which rigid body modes ────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

def draw_plate(ax, width=1.0, height=1.0):
    ax.add_patch(patches.FancyBboxPatch(
        (0, 0), width, height,
        boxstyle='round,pad=0.02', facecolor='lightblue', edgecolor='navy', linewidth=1.5
    ))
    ax.set_xlim(-0.3, 1.3); ax.set_ylim(-0.3, 1.3); ax.set_aspect('equal')
    ax.axis('off')

# Plot 1: Uniaxial tension — correct
ax = axes[0]
draw_plate(ax)
ax.annotate('UX=0 left face', xy=(0, 0.5), xytext=(-0.25, 0.5),
             fontsize=8, ha='right', color='darkgreen',
             arrowprops=dict(arrowstyle='->', color='darkgreen'))
ax.annotate('UX=δ right face', xy=(1.0, 0.5), xytext=(1.2, 0.5),
             fontsize=8, ha='left', color='red',
             arrowprops=dict(arrowstyle='->', color='red'))
ax.plot(0, 0, 'ko', markersize=8)  # bottom-left pin
ax.annotate('UY=0 here', xy=(0, 0), xytext=(0.05, -0.22), fontsize=7, color='purple')
ax.set_title('Uniaxial tension\n(correct)', fontsize=9)

# Plot 2: Biaxial tension
ax = axes[1]
draw_plate(ax)
ax.plot(0, 0, 'ks', markersize=10)  # corner pin
ax.annotate('(0,0)\nUX=UY=0', xy=(0,0), xytext=(0.05, -0.25), fontsize=7, color='purple')
ax.annotate('UX=δ', xy=(1,0.5), xytext=(1.15, 0.5), fontsize=8, color='red',
             arrowprops=dict(arrowstyle='->', color='red'))
ax.annotate('UY=δ', xy=(0.5,1), xytext=(0.5, 1.15), fontsize=8, color='red',
             arrowprops=dict(arrowstyle='->', color='red'))
ax.set_title('Biaxial tension\n(correct)', fontsize=9)

# Plot 3: Pure shear
ax = axes[2]
draw_plate(ax)
ax.annotate('Bottom: ALL=0', xy=(0.5, 0), xytext=(0.5, -0.25), fontsize=8,
             ha='center', color='darkgreen',
             arrowprops=dict(arrowstyle='->', color='darkgreen'))
ax.annotate('Top: UX=δ, UY=0', xy=(0.5, 1), xytext=(0.5, 1.2), fontsize=8,
             ha='center', color='red',
             arrowprops=dict(arrowstyle='->', color='red'))
ax.set_title('Simple shear\n(pure mode)', fontsize=9)

plt.suptitle('Dirichlet BC Presets — Kinematic Support Diagrams', fontsize=11)
plt.tight_layout(); plt.show()

print('Physical note: biaxial preset pins only the (0,0) corner — minimal constraint')
print('that removes all 2D rigid body modes (2 translations + 1 rotation)')
print('without over-constraining the Poisson contraction.')

---

## Part 3: Periodic Boundary Conditions (Unit Cell Analysis)

### When to use periodic BCs

When your structure repeats infinitely in one or more directions (metamaterials,
woven composites, origami tessellations), you can analyze just ONE unit cell with
periodic BCs instead of a large finite array.

### Mathematics

For a unit cell with periodicity vector $\mathbf{L}$ (length $L$ along X):

$$\mathbf{u}(\mathbf{x} + \mathbf{L}) = \mathbf{u}(\mathbf{x}) + \bar{\boldsymbol{\varepsilon}} \cdot \mathbf{L}$$

where $\bar{\boldsymbol{\varepsilon}}$ is the macroscopic strain imposed on the cell.

In MAPDL, this is a **constraint equation** between matched node pairs:

$$u_i^{\text{hi}} - u_i^{\text{lo}} = \bar{\varepsilon}_{i1} L_x \quad \forall i$$

For zero macroscopic strain ($\bar{\varepsilon} = 0$): simply $u_i^{\text{hi}} = u_i^{\text{lo}}$
— the displacements on opposite faces are equal.  This is implemented with `CP` (coupled DOF).

### Requirement: matching mesh on periodic faces

The node-pairing algorithm matches nodes by their coordinates in the non-periodic
directions.  If the meshes on the two faces are not identical (same node positions
in Y and Z when periodic in X), the pairing will fail.

**Always use a mapped or swept mesh for unit cell analysis.**  nTopology's
hex-dominant mesher and MAPDL's `LESIZE` + `MSHKEY,1` ensure face-matched meshes.

In [ ]:
# ── Visualize: periodic BC node pairing ───────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

# Draw unit cell
rect = patches.Rectangle((0, 0), 1.0, 1.0, linewidth=2, edgecolor='navy', facecolor='lightblue', alpha=0.4)
ax.add_patch(rect)

# Draw node pairs
n_pairs = 5
ys = np.linspace(0.1, 0.9, n_pairs)

for i, y in enumerate(ys):
    # Lo face (x=0)
    ax.plot(0, y, 'go', markersize=10)
    ax.annotate(f'lo_{i}', (0, y), xytext=(-0.15, y), fontsize=8, ha='right', color='green')
    
    # Hi face (x=1)
    ax.plot(1.0, y, 'rs', markersize=10)
    ax.annotate(f'hi_{i}', (1.0, y), xytext=(1.15, y), fontsize=8, ha='left', color='red')
    
    # Coupling arrow
    ax.annotate('', xy=(1.0, y), xytext=(0, y),
                 arrowprops=dict(arrowstyle='<->', color='purple', lw=1.2, linestyle='dashed'))

ax.set_xlim(-0.3, 1.3); ax.set_ylim(-0.1, 1.1); ax.set_aspect('equal')
ax.set_title('Periodic BC: node pairing on lo (green) and hi (red) faces', fontsize=10)
ax.set_xlabel('X'); ax.set_ylabel('Y')
ax.text(0.5, -0.08, 'Constraint: CP, nset, DOF, lo_node, hi_node', ha='center', fontsize=9, color='purple')
ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

print('Physical meaning: the displacement at hi_i equals that at lo_i for the same Y coord.')
print('This enforces u(x + L_x) = u(x) — the periodic boundary condition.')

In [ ]:
# ── Show the apply_periodic function signature and implementation ──────────
from ams.mapdl.boundary import apply_periodic
import inspect

# Print the function's docstring
print(apply_periodic.__doc__)

---

## Part 4: Neumann BCs (Tractions and Pressures)

### Sign convention for pressure loads in ANSYS

ANSYS defines positive pressure as **compressive** (acting inward on the face).
For a **tensile** load on the right face (+X direction):
- The traction is $\mathbf{t} = +\sigma_{far} \hat{\mathbf{x}}$ (pointing in +X)
- The ANSYS pressure is $p = -\sigma_{far}$ (negative = tensile)

**This is one of the most common mistakes in structural FEA setup.**

### Kirsch stress concentration factor

For a circular hole in an infinite plate under uniaxial tension $\sigma_{far}$:

$$\sigma_{\theta\theta}(r, \theta) = \frac{\sigma_{far}}{2} \left(1 - \frac{a^2}{r^2}\right) + \frac{\sigma_{far}}{2} \left(1 + \frac{3a^4}{r^4} - \frac{4a^2}{r^2}\right) \cos(2\theta)$$

At $r = a$ (hole edge), $\theta = 90°$:
$$\sigma_{\theta\theta}^{\text{max}} = 3 \sigma_{far} \quad \Rightarrow \quad K_t = 3$$

This is the famous **stress concentration factor** of 3 for a circular hole.
Your FEA should reproduce this within $< 5\%$ with a refined mesh near the hole.

In [ ]:
# ── Kirsch solution: stress distribution around a circular hole ────────────
import numpy as np
import matplotlib.pyplot as plt

a = 0.01       # hole radius (m)
sigma_far = 100e6  # far-field stress (100 MPa)

# Radial distribution along theta = 90 (peak stress line)
r = np.linspace(a, 5*a, 500)
theta = np.pi / 2

# Kirsch solution
sigma_rr  = sigma_far/2 * (1 - a**2/r**2) + sigma_far/2 * (1 - 4*a**2/r**2 + 3*a**4/r**4) * np.cos(2*theta)
sigma_tt  = sigma_far/2 * (1 + a**2/r**2) - sigma_far/2 * (1 + 3*a**4/r**4) * np.cos(2*theta)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: stress vs r/a
ax = axes[0]
ax.plot(r/a, sigma_tt/sigma_far, 'b-', linewidth=2, label='σ_θθ (hoop, θ=90°)')
ax.plot(r/a, sigma_rr/sigma_far, 'r--', linewidth=2, label='σ_rr (radial, θ=90°)')
ax.axhline(1, color='gray', linestyle=':', label='σ_far')
ax.axhline(3, color='purple', linestyle=':', alpha=0.7, label='Kt=3 at r/a=1')
ax.set_xlabel('r/a (normalized radius)'); ax.set_ylabel('σ/σ_far')
ax.set_title('Kirsch Solution — Stress Decay')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Right: circumferential stress around hole edge (r=a)
ax = axes[1]
thetas = np.linspace(0, 2*np.pi, 360)
sigma_tt_edge = sigma_far * (1 - 2*np.cos(2*thetas))
ax.polar if False else ax  # skip polar for simplicity
ax.plot(np.degrees(thetas), sigma_tt_edge/sigma_far, 'b-', linewidth=2)
ax.axhline(3, color='red', linestyle='--', label='Kt=3 (max)')
ax.axhline(-1, color='green', linestyle='--', label='Kt=-1 (min, compressive)')
ax.set_xlabel('θ (degrees)'); ax.set_ylabel('σ_θθ/σ_far')
ax.set_title('Circumferential Stress at Hole Edge (r=a)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print(f'\nKirsch peak stress: Kt = 3.00')
print(f'At σ_far = {sigma_far/1e6:.0f} MPa:')
print(f'  σ_max = {3 * sigma_far/1e6:.0f} MPa (at θ = 90°, r = a)')
print(f'  σ_min = {-1 * sigma_far/1e6:.0f} MPa (at θ = 0°, r = a) — compressive!')

---

## Summary

| BC Type | Equation | Physical Meaning | MAPDL |
|---------|----------|------------------|-------|
| Dirichlet | u = ū | Fixed/prescribed motion | `D, ALL, DOF, value` |
| Neumann | σ·n = t | Pressure/traction | `SF, ALL, PRES, value` |
| Robin | σ·n = −k_s u | Elastic foundation | SURF154 elements |
| Periodic | u(x+L)=u(x) | Unit cell | `CP, set, DOF, n1, n2` |
| Symmetry | u_n = 0 | Half-model | `D, ALL, UX/UY/UZ, 0` |

**Next:** `05_solver_selection_and_strategy.ipynb`